# Datamine COMPBR Process Example

This notebook demonstrates how to configure and run the **`compbr`** process wrapper in `dmstudio`.

## Process Description

## COMPBR

Composites drillhole data over horizontal benches, with additional computation of a recovered grade and recovered fraction for a specified field at a given cut-off.

The **COMPBR** process is a modification of the [COMPBE](<compbe.md>) process, and composites in an identical way. The difference comes in the recovered grade and fraction calculations. For the specified field * **VALUE** , the weighted **FRACTION** of each composite above the given parameter @**CUTOFF** is computed, together with the mean grade **REC.VAL** of this fraction. These values relate to those that could be achieved with selective mining within benches at the specified cut-off.

Further details of the compositing process are given in the [COMPBE](<compbe.md>) description.

The file must be in the order of **BHID** and **FROM** (sorted in drillhole order in increasing downhole distance). This is the order output from the [DESURV](<desurv.md>) process.

* **Note** (A progress message is displayed for every 500 samples read.):

### Weighting by density

If a density field exists in the file then this may be used too for density weighted compositing. The density field is defined as the optional field *DENSITY. If any density value is absent data then the default density value will be used.

### Weighting by core loss or recovery

To include the effects of core loss, the user may specify one of two optional fields * **CORELOSS** (core loss as a percentage) or * **COREREC** (core recovery as a percentage) to be used during compositing. The lost portion of the core will be taken into account and used in compositing. The actual treatment depends on the optional @**LOSS** parameter. If @**LOSS** <=0 (default) then the lost part of the core will be assumed to have exactly the same grades, properties etc. as the recovered part; in other words, the core loss is ignored. If however @**LOSS** = 1 then the lost core will be assumed to have default grades, density, properties etc. which will be averaged with the recovered core values.

If @LOSS>=2 then the lost core will be treated as cavity (zero density and grades) so that the grade of the total sample is effectively reduced by the cavity.

### Input Files:

* **in** (Drillhole):
  Sample data file, sorted on BHID and FROM. Expects fields BHID, FROM, TO, LENGTH, X, Y,

## Z, A0, B0.

  Required=Yes

### Output Files:

* **out** (Drillhole):
  Composite file. Will include implicit field CUTOFF and explicit fields REC.VAL and
  FRACTION for recovered values.
  Required=Yes

### Fields:

* **value** (Numeric : IN):
  Field for recovered grade computations.
  Default=Undefined
  Required=Yes

* **bhid** (Any : IN):
  Drillhole identifier.
  Default=BHID
  Required=No

* **from** (Numeric : IN):
  Downhole distance to sample top.
  Default=FROM
  Required=No

* **to** (Numeric : IN):
  Downhole distance to sample bottom.
  Default=TO
  Required=No

* **density** (Numeric : IN):
  If present, composites will be density-weighted
  Default=DENSITY
  Required=No

* **coreloss** (Numeric : IN):
  If present, will be taken as percentage core loss, and treated according to the LOSS
  parameter.
  Default=CORELOSS
  Required=No

* **corerec** (Numeric : IN):
  If present, will be taken as percentage core recovery, (100-core loss) and treated
  according to the LOSS parameter.
  Default=COREREC
  Required=No

* **zone** (Any : IN):
  Name of field for compositing within. (may be numeric or up to 4 character alpha). This
  field must exist in the IN and will be copied to the OUT file. If specified then new
  composites will be created each time the value of ZONE changes.
  Default=Undefined
  Required=No

### Parameters:

* **interval**:
  Bench height.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **cutoff**:
  Cutoff to be applied to VALUE.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=Yes

* **mingap**:
  Gap length to be ignored. The default gap is calculated as 0.05 INTERVAL. This default
  value is applied if the parameter is not specified, or if the value is specified as <=0.
  A gap of exactly zero is not permitted. If you want the composite to be split at every
  gap, use a very small value for MAXGAP eg 0.0001.
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **maxgap**:
  Gap length for termination of composite (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **elev**:
  Reference bench elevation (0).
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **mincomp**:
  Minimum composite length [0.5 INTERVAL].
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **maxcomp**:
  Maximum composite length [2.0 INTERVAL].
  Range=Undefined
  Values=Undefined
  Default=Undefined
  Required=No

* **loss**:
  If core loss or core recovery field is present, controls how it is handled: Options: 0:
  Treat loss as part of sample.; 1: Treat loss as default values.; 2: Treat as cavity
  [zero density and grades]
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

* **print**:
  >2 to display each composite and output file DD (0).
  Range=0,2
  Values=0,1,2
  Default=0
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('compbr')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute compbr
print("Running compbr...")
dm_cmd.compbr(
    in_i='t_assays',  # required
    out_o='t_compbr_out',  # required
    value_f='optional',  # required
    zone_f='optional',  # required
    interval_p='required_val',  # required
    cutoff_p='required_val',  # required
    # bhid_f='BHID',  # optional
    # from_f='FROM',  # optional
    # to_f='TO',  # optional
    # density_f='optional',  # optional
    # coreloss_f=['optional_field'],  # optional
    # corerec_f='optional',  # optional
    # mingap_p='optional',  # optional
    # maxgap_p=0,  # optional
    # elev_p=0,  # optional
    # mincomp_p='optional',  # optional
    # maxcomp_p='optional',  # optional
    # loss_p=0,  # optional
    # print_p=0,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("compbr execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_compbr_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")